In [2]:
import trimesh
import pyvista as pv
import numpy as np

PolyData (0x73fcfdb8c160)
  N Cells:    136
  N Points:   60
  N Strips:   0
  X Bounds:   -2.949e-17, 2.515e-02
  Y Bounds:   -9.525e-03, 3.175e-03
  Z Bounds:   -1.454e-10, 4.191e-02
  N Arrays:   0


In [4]:
# -----------------------------
# Step 1: Load your STL magnetic core
# -----------------------------
mesh = trimesh.load("core.stl")  # replace with your STL path
vertices = mesh.vertices
faces = mesh.faces

# PyVista expects faces as [n, v0, v1, v2, ...] for each triangle
faces_pv = np.hstack([np.full((faces.shape[0], 1), 3), faces]).astype(np.int64)
faces_pv = faces_pv.flatten()

pv_mesh = pv.PolyData(vertices, faces_pv)

In [7]:
import magpylib as mag
import trimesh
import pyvista as pv
import numpy as np

# -----------------------------
# Step 1: Load STL core
# -----------------------------
mesh = trimesh.load("core.stl")
vertices = mesh.vertices
faces = mesh.faces

faces_pv = np.hstack([np.full((faces.shape[0], 1), 3), faces]).astype(np.int64)
faces_pv = faces_pv.flatten()
pv_mesh = pv.PolyData(vertices, faces_pv)

# -----------------------------
# Step 2: Define a circular coil (magpylib 4+)
# -----------------------------
coil = mag.magnet.Circular(
    current=10,    # Amperes
    diameter=0.05, # meters
    turns=10
)
coil.position = (0, 0, 0)    # center of core window
coil.orientation = (0, 0, 1) # along z-axis

# -----------------------------
# Step 3: Observation points
# -----------------------------
x = np.linspace(-0.1, 0.1, 20)
y = np.linspace(-0.1, 0.1, 20)
z = np.linspace(-0.05, 0.05, 10)
X, Y, Z = np.meshgrid(x, y, z)
points = np.vstack([X.ravel(), Y.ravel(), Z.ravel()]).T

# -----------------------------
# Step 4: Compute magnetic field
# -----------------------------
B = coil.getB(points)
B_magnitude = np.linalg.norm(B, axis=1)

# -----------------------------
# Step 5: Visualize
# -----------------------------
point_cloud = pv.PolyData(points)
point_cloud["B_magnitude"] = B_magnitude

plotter = pv.Plotter()
plotter.add_mesh(pv_mesh, color="lightgray", opacity=0.5, show_edges=True)
plotter.add_points(point_cloud, scalars="B_magnitude",
                   render_points_as_spheres=True, point_size=5,
                   cmap="viridis")
plotter.add_axes()
plotter.show()


AttributeError: module 'magpylib.magnet' has no attribute 'Circular'